In [0]:
%run ../configs/log_configs

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType,
    IntegerType, TimestampType
)
from datetime import datetime
import json

today = datetime.now()
run_date = today.strftime("%Y-%m-%d")

In [0]:
dbutils.widgets.text("environment",   "dev")
dbutils.widgets.text("layer",         "gold")
dbutils.widgets.text("notebook_name", "04_data_quality_scorecard")
dbutils.widgets.text("pipeline_name", "BIB_Data_Processing")

environment   = dbutils.widgets.get("environment")
layer         = dbutils.widgets.get("layer")
notebook_name = dbutils.widgets.get("notebook_name")
pipeline_name = dbutils.widgets.get("pipeline_name")

print(f"Environment : {environment}")
print(f"Run Date    : {run_date}")

In [0]:
# ─────────────────────────────────────────────
# Cell 3: Generic DQ Check Functions
# Run against any Delta table
# ─────────────────────────────────────────────

def check_null_rates(df, table_name):
    """
    Returns null % for every column.
    Flags columns with null rate > 5% as WARNING
    and > 20% as FAIL.
    """
    total = df.count()
    if total == 0:
        return []

    results = []
    for col_name in df.columns:
        null_count = df.filter(
            F.col(col_name).isNull()
        ).count()
        null_pct = round(
            (null_count / total) * 100, 2
        )

        if null_pct > 20:
            status = "FAIL"
        elif null_pct > 5:
            status = "WARNING"
        else:
            status = "PASS"

        results.append({
            "table_name":  table_name,
            "check_type":  "null_rate",
            "column_name": col_name,
            "metric_value": null_pct,
            "threshold":   "FAIL>20%, WARN>5%",
            "status":      status,
            "detail": (
                f"{null_count} nulls out of "
                f"{total} rows ({null_pct}%)"
            )
        })

    return results


def check_duplicate_rate(df, table_name, primary_key):
    """
    Checks for duplicate primary keys.
    Any duplicates = FAIL.
    """
    total      = df.count()
    distinct   = df.select(primary_key).distinct().count()
    dup_count  = total - distinct
    dup_pct    = round((dup_count / total) * 100, 2) if total > 0 else 0

    status = "FAIL" if dup_count > 0 else "PASS"

    return [{
        "table_name":   table_name,
        "check_type":   "duplicate_rate",
        "column_name":  primary_key,
        "metric_value": dup_pct,
        "threshold":    "FAIL if any duplicates",
        "status":       status,
        "detail": (
            f"{dup_count} duplicate {primary_key} "
            f"values found"
        )
    }]


def check_referential_integrity(
    child_df, parent_df,
    child_table, parent_table,
    join_col
):
    """
    Checks that all values in child.join_col
    exist in parent.join_col.
    Orphan rate > 0 = WARNING,
    orphan rate > 10% = FAIL.
    """
    total = child_df.select(join_col).count()
    if total == 0:
        return []

    parent_keys = parent_df.select(
        join_col
    ).distinct()

    orphans = child_df.join(
        parent_keys,
        on=join_col,
        how="left_anti"
    ).count()

    orphan_pct = round(
        (orphans / total) * 100, 2
    )

    if orphan_pct > 10:
        status = "FAIL"
    elif orphan_pct > 0:
        status = "WARNING"
    else:
        status = "PASS"

    return [{
        "table_name": child_table,
        "check_type": "referential_integrity",
        "column_name": join_col,
        "metric_value": orphan_pct,
        "threshold": (
            f"FAIL>10%, WARN>0% "
            f"({child_table}.{join_col} "
            f"→ {parent_table})"
        ),
        "status": status,
        "detail": (
            f"{orphans} records in {child_table} "
            f"have no matching {join_col} "
            f"in {parent_table}"
        )
    }]


def check_freshness(df, table_name, date_col):
    """
    Checks how recent the latest record is.
    > 1 day old = WARNING, > 3 days = FAIL.
    """
    try:
        latest = df.agg(
            F.max(F.col(date_col))
        ).collect()[0][0]

        if latest is None:
            return [{
                "table_name":   table_name,
                "check_type":   "freshness",
                "column_name":  date_col,
                "metric_value": -1,
                "threshold":    "FAIL>3d, WARN>1d",
                "status":       "FAIL",
                "detail":       "No data found in table"
            }]

        from datetime import timezone
        now = datetime.now()
        # Handle both date and timestamp types
        if hasattr(latest, 'date'):
            days_old = (now.date() - latest.date()).days
        else:
            days_old = (now.date() - latest).days

        if days_old > 3:
            status = "FAIL"
        elif days_old > 1:
            status = "WARNING"
        else:
            status = "PASS"

        return [{
            "table_name":   table_name,
            "check_type":   "freshness",
            "column_name":  date_col,
            "metric_value": float(days_old),
            "threshold":    "FAIL>3 days, WARN>1 day",
            "status":       status,
            "detail": (
                f"Latest record: {latest}, "
                f"{days_old} day(s) old"
            )
        }]
    except Exception as e:
        return [{
            "table_name":   table_name,
            "check_type":   "freshness",
            "column_name":  date_col,
            "metric_value": -1,
            "threshold":    "FAIL>3d, WARN>1d",
            "status":       "WARNING",
            "detail":       f"Freshness check failed: {str(e)}"
        }]


def check_row_count(df, table_name, min_expected):
    """
    Checks table has minimum expected rows.
    Below min = FAIL.
    """
    count  = df.count()
    status = "FAIL" if count < min_expected else "PASS"

    return [{
        "table_name":   table_name,
        "check_type":   "row_count",
        "column_name":  "N/A",
        "metric_value": float(count),
        "threshold":    f"MIN {min_expected} rows",
        "status":       status,
        "detail": (
            f"{count} rows found, "
            f"minimum expected: {min_expected}"
        )
    }]

In [0]:
# ─────────────────────────────────────────────
# Cell 4: Run DQ Checks on All Silver + Gold Tables
# ─────────────────────────────────────────────

all_results = []

print("=" * 50)
print("Running Data Quality Checks")
print("=" * 50)

# ── Load tables ───────────────────────────────
try:
    txn         = spark.read.table("bib_catalog.silver.transactions")
    customers   = spark.read.table("bib_catalog.silver.customers").filter(F.col("scd_is_current") == True)
    campaigns   = spark.read.table("bib_catalog.silver.campaign_interactions")
    digital     = spark.read.table("bib_catalog.silver.digital_activity")
    c360        = spark.read.table("bib_catalog.gold.customer_360")
    txn_intel   = spark.read.table("bib_catalog.gold.transaction_intelligence")
    camp_perf   = spark.read.table("bib_catalog.gold.campaign_performance")

    print("All tables loaded successfully\n")
except Exception as e:
    print(f"Table load error: {str(e)}")
    raise

# ────────────────────────────────────────────
# Silver: Transactions
# ────────────────────────────────────────────
print("Checking: silver.transactions")
all_results += check_row_count(
    txn, "silver.transactions", min_expected=100
)
all_results += check_duplicate_rate(
    txn, "silver.transactions", "txn_id"
)
all_results += check_null_rates(
    txn.select(
        "txn_id", "customer_id",
        "amount", "txn_datetime", "status"
    ),
    "silver.transactions"
)
all_results += check_freshness(
    txn, "silver.transactions", "txn_datetime"
)

# ────────────────────────────────────────────
# Silver: Customers
# ────────────────────────────────────────────
print("Checking: silver.customers")
all_results += check_row_count(
    customers, "silver.customers", min_expected=100
)
all_results += check_duplicate_rate(
    customers, "silver.customers", "customer_id"
)
all_results += check_null_rates(
    customers.select(
        "customer_id", "full_name",
        "email", "branch_id", "manager_id"
    ),
    "silver.customers"
)

# ────────────────────────────────────────────
# Silver: Campaign Interactions
# ────────────────────────────────────────────
print("Checking: silver.campaign_interactions")
all_results += check_row_count(
    campaigns, "silver.campaign_interactions",
    min_expected=100
)
all_results += check_duplicate_rate(
    campaigns, "silver.campaign_interactions",
    "interaction_id"
)
all_results += check_freshness(
    campaigns, "silver.campaign_interactions",
    "interaction_datetime"
)

# ────────────────────────────────────────────
# Referential Integrity Checks
# ────────────────────────────────────────────
print("Checking: referential integrity")
all_results += check_referential_integrity(
    txn, customers,
    "silver.transactions",
    "silver.customers",
    "customer_id"
)
all_results += check_referential_integrity(
    campaigns, customers,
    "silver.campaign_interactions",
    "silver.customers",
    "customer_id"
)
all_results += check_referential_integrity(
    digital, customers,
    "silver.digital_activity",
    "silver.customers",
    "customer_id"
)

# ────────────────────────────────────────────
# Gold: Customer 360
# ────────────────────────────────────────────
print("Checking: gold.customer_360")
all_results += check_row_count(
    c360, "gold.customer_360", min_expected=100
)
all_results += check_null_rates(
    c360.select(
        "customer_id", "full_name",
        "zone", "region", "engagement_score"
    ),
    "gold.customer_360"
)

# ────────────────────────────────────────────
# Gold: Transaction Intelligence
# ────────────────────────────────────────────
print("Checking: gold.transaction_intelligence")
all_results += check_row_count(
    txn_intel, "gold.transaction_intelligence",
    min_expected=10
)
all_results += check_null_rates(
    txn_intel.select(
        "txn_date", "txn_type",
        "txn_count", "total_amount"
    ),
    "gold.transaction_intelligence"
)

# ────────────────────────────────────────────
# Gold: Campaign Performance
# ────────────────────────────────────────────
print("Checking: gold.campaign_performance")
all_results += check_row_count(
    camp_perf, "gold.campaign_performance",
    min_expected=5
)

print(f"\nTotal DQ checks run: {len(all_results)}")

In [0]:
# ─────────────────────────────────────────────
# Cell 5: Write DQ Scorecard to Unity Catalog
# ─────────────────────────────────────────────

schema = StructType([
    StructField("table_name",   StringType(),  True),
    StructField("check_type",   StringType(),  True),
    StructField("column_name",  StringType(),  True),
    StructField("metric_value", DoubleType(),  True),
    StructField("threshold",    StringType(),  True),
    StructField("status",       StringType(),  True),
    StructField("detail",       StringType(),  True),
    StructField("run_date",     StringType(),  True),
    StructField("environment",  StringType(),  True),
    StructField("checked_at",   TimestampType(),True)
])

now_ts = datetime.now()

rows = [
    (
        r["table_name"],
        r["check_type"],
        r["column_name"],
        float(r["metric_value"]),
        r["threshold"],
        r["status"],
        r["detail"],
        run_date,
        environment,
        now_ts
    )
    for r in all_results
]

scorecard_df = spark.createDataFrame(rows, schema)

# ── Ensure schema exists ───────────────────
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS bib_catalog.gold"
)

# ── Write to Unity Catalog ─────────────────
(
    scorecard_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("bib_catalog.gold.dq_scorecard")
)
print("DQ Scorecard written to Unity Catalog")

# ── Write to ADLS ──────────────────────────
adls_dq_path = (
    f"abfss://dev@{storage_account}"
    f".dfs.core.windows.net/gold/dq_scorecard/"
)
(
    scorecard_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(adls_dq_path)
)
print("DQ Scorecard written to ADLS")

# ── Print summary ──────────────────────────
print("\n" + "=" * 50)
print("DQ SCORECARD SUMMARY")
print("=" * 50)
scorecard_df.groupBy("status").count().show()
print("\nFailed / Warning checks:")
scorecard_df.filter(
    F.col("status").isin(["FAIL", "WARNING"])
).select(
    "table_name", "check_type",
    "column_name", "metric_value", "detail"
).show(truncate=False)

In [0]:
%pip install google-generativeai --quiet

In [0]:
# ─────────────────────────────────────────────
# Cell 2c: LLM Connection — Cached
# ─────────────────────────────────────────────
import google.generativeai as genai

# ── Configure once ─────────────────────────
genai.configure(api_key=gemini_api_key)

# ── Model name from Key Vault ──────────────
# No hardcoding — change model in vault only
_LLM_MODEL = genai.GenerativeModel(
    model_name=gemini_model_name,
    generation_config={
        "temperature":       0.2,
        "max_output_tokens": 500,
        "top_p":             0.8
    }
)

print(f"LLM model cached    : {gemini_model_name}")
print(f"API configured      : Gemini")


def get_llm_insight(prompt: str) -> str:
    try:
        response = _LLM_MODEL.generate_content(prompt)
        return response.text.strip()
    except Exception as llm_err:
        print(f"LLM call failed: {str(llm_err)}")
        return f"Insight unavailable: {str(llm_err)}"

In [0]:
# ─────────────────────────────────────────────
# Cell 2e: AI Insights Table Writer
# Writes all LLM insights to a Gold Delta table
# Queryable via SQL — shown in Power BI + Streamlit
# ─────────────────────────────────────────────
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType
)

def write_ai_insight(
    table_name, insight_text,
    insight_type, source_table
):
    """
    Persists LLM insight to
    bib_catalog.gold.ai_insights Delta table.
    Appends — full history of daily insights kept.
    """
    schema = StructType([
        StructField("insight_id",    StringType(),   True),
        StructField("table_name",    StringType(),   True),
        StructField("insight_type",  StringType(),   True),
        StructField("source_table",  StringType(),   True),
        StructField("insight_text",  StringType(),   True),
        StructField("environment",   StringType(),   True),
        StructField("generated_at",  TimestampType(),True)
    ])

    import uuid
    from datetime import datetime

    row = [(
        str(uuid.uuid4()),
        table_name,
        insight_type,
        source_table,
        insight_text,
        environment,
        datetime.now()
    )]

    insight_df = spark.createDataFrame(row, schema)

    # ── Ensure schema exists ───────────────
    spark.sql(
        "CREATE SCHEMA IF NOT EXISTS "
        "bib_catalog.gold"
    )

    # ── Append to insights table ───────────
    (
        insight_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable("bib_catalog.gold.ai_insights")
    )

    # ── Also write to ADLS ─────────────────
    adls_insights_path = get_adls_path(
        environment, layer, "ai_insights/"
    )
    (
        insight_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .save(adls_insights_path)
    )

    print(
        f"AI insight written  : "
        f"{insight_type} for {table_name}"
    )

In [0]:
# ─────────────────────────────────────────────
# Cell 6: AI-Generated DQ Report
# Feeds scorecard summary to Gemini
# Produces an executive data quality narrative
# ─────────────────────────────────────────────

# ── Compute summary stats for LLM ─────────
summary = scorecard_df.groupBy("status").count().collect()
status_dict = {row["status"]: row["count"] for row in summary}

pass_count    = status_dict.get("PASS", 0)
warn_count    = status_dict.get("WARNING", 0)
fail_count    = status_dict.get("FAIL", 0)
total_checks  = pass_count + warn_count + fail_count
pass_rate     = round((pass_count / total_checks) * 100, 1) if total_checks > 0 else 0

# ── Collect failed/warning details ────────
issues = scorecard_df.filter(
    F.col("status").isin(["FAIL", "WARNING"])
).select(
    "table_name", "check_type",
    "column_name", "detail"
).collect()

issue_lines = "\n".join([
    f"  [{r['status'] if 'status' in r.asDict() else '?'}] "
    f"{r['table_name']} | {r['check_type']} | "
    f"{r['column_name']}: {r['detail']}"
    for r in issues
])

# Re-collect with status for the prompt
issues_with_status = scorecard_df.filter(
    F.col("status").isin(["FAIL", "WARNING"])
).select(
    "table_name", "check_type",
    "column_name", "status", "detail"
).collect()

issue_lines = "\n".join([
    f"  [{r['status']}] {r['table_name']} | "
    f"{r['check_type']} | {r['column_name']}: "
    f"{r['detail']}"
    for r in issues_with_status
])

# ── Build LLM prompt ──────────────────────
prompt = f"""
You are a senior data quality analyst for Bharat Indus Bank.
Today's automated data quality scorecard results:

Run Date    : {run_date}
Total Checks: {total_checks}
PASS        : {pass_count} ({pass_rate}%)
WARNING     : {warn_count}
FAIL        : {fail_count}

Issues Found:
{issue_lines if issue_lines else "None — all checks passed!"}

Write a concise 5-6 line executive data quality report:
1. Overall data health status (Good / At Risk / Critical)
2. Key issues and which tables are affected
3. Business impact of any failures
4. One specific recommended action
5. Whether the pipeline data can be trusted for reporting today

Keep it professional, clear, and actionable.
"""

# ── Call LLM ──────────────────────────────
print("Generating AI data quality narrative...")
dq_narrative = get_llm_insight(prompt)

print("\n" + "=" * 50)
print("AI DATA QUALITY REPORT")
print("=" * 50)
print(dq_narrative)
print("=" * 50)

# ── Write AI narrative to insights table ──
write_ai_insight(
    table_name="dq_scorecard",
    insight_text=dq_narrative,
    insight_type="Data Quality Report",
    source_table="bib_catalog.gold.dq_scorecard"
)
print("\nAI DQ narrative written to ai_insights table")

In [0]:
# ─────────────────────────────────────────────
# Cell 7: Log to Pipeline Logs
# ─────────────────────────────────────────────

overall_status = (
    "FAILED"   if fail_count  > 0 else
    "WARNING"  if warn_count  > 0 else
    "SUCCESS"
)

log_pipeline_event(
    notebook_name=notebook_name,
    pipeline_name=pipeline_name,
    source_name="dq_scorecard",
    layer=layer,
    status=overall_status,
    records_processed=total_checks,
    message=(
        f"DQ Scorecard: {total_checks} checks | "
        f"PASS={pass_count} WARN={warn_count} "
        f"FAIL={fail_count}"
    ),
    environment=environment
)

print(
    f"\nFinal DQ Status : {overall_status}"
)
print(
    f"Checks          : {total_checks} total | "
    f"{pass_count} pass | "
    f"{warn_count} warn | "
    f"{fail_count} fail"
)

# Raise if critical failures found
if fail_count > 0:
    raise Exception(
        f"Data Quality FAILED: {fail_count} checks failed. "
        f"See bib_catalog.gold.dq_scorecard for details."
    )

In [0]:
%sql
-- See today's scorecard
SELECT
    table_name,
    check_type,
    column_name,
    metric_value,
    status,
    detail
FROM bib_catalog.gold.dq_scorecard
WHERE run_date = CURRENT_DATE()
ORDER BY
    CASE status
        WHEN 'FAIL'    THEN 1
        WHEN 'WARNING' THEN 2
        ELSE 3
    END,
    table_name;

In [0]:
%sql
-- See today's AI narrative
SELECT insight_text, generated_at
FROM bib_catalog.gold.ai_insights
WHERE insight_type = 'Data Quality Report'
ORDER BY generated_at DESC
LIMIT 1;